# The Photoelectric Effect

## Purpose
In this lab you will:
- **Chemistry**: Analyze experimental photoelectric spectra to extract the stopping potential, determine Planck's constant experimentally, and understand energy quantization in light.
- **Coding**: Load and manipulate tabular data with pandas, write reusable functions for data processing, and perform linear regression to extract physical parameters.

**Real-world connection**: The photoelectric effect provided key evidence for the quantum nature of light and is the basis for modern photodetectors, solar cells, and photoelectron spectroscopy (XPS/UPS).

## Estimated Time: 60-90 minutes

## Success Criteria
- [ ] Successfully load and plot photoelectric spectra from CSV files
- [ ] Extract the stopping potential from a spectrum using linear regression
- [ ] Write a reusable `process_spectrum()` function that works on any spectrum file
- [ ] Determine Planck's constant from your data and compare to the accepted value
- [ ] Calculate the work function of calcium

---
# LIBRARIES

Execute this cell before proceeding.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

---
# Warmup: DataFrames and Tabular Data
`10 points`

Scientific data is often **tabular**: rows represent observations and columns represent variables. The `pandas` library provides the `DataFrame` object for efficient tabular data manipulation.

In this warmup, you will construct a simple DataFrame, inspect its structure, and perform basic column operations.

### CODE: Create a DataFrame

Construct a pandas DataFrame that represents the given data.

**Requirements:**
- One row per student
- One column for each recorded quantity
- Use clear, descriptive column labels

Display the DataFrame as the final line of the cell.

In [ ]:
students    = ['Bob', 'Alice', 'Giancarlo']
test_scores = [95, 85, 75]
gpa         = [3.0, 2.0, 5.0]

# Create a DataFrame that represents this data.
# Use clear, meaningful column labels.
# Display the DataFrame as the final line of this cell.

### Question 1 (2 points)

Suppose one entry in your numerical column were missing (`NaN`). What would pandas do when computing a new scaled column (e.g., `df['GPA'] * 5`)?

A. Raise an error  
B. Skip the row entirely  
C. Propagate the missing value  
D. Replace it with zero  

**Explain your choice.**

### CODE: Column Operations

Using your DataFrame:
- Multiply the GPA column by 5
- Store the result in a **new** column (do not overwrite the original)
- Display the updated DataFrame

In [ ]:
# Add a new column based on an existing column.
# Do not overwrite the original GPA column.

### Question 2 (2 points)

Which statement best describes what happens when you multiply a DataFrame column by a number?

A. The operation is applied element-by-element  
B. Only the first value is modified  
C. The column must be looped over manually  
D. The DataFrame is converted to a list  

**Explain your reasoning.**

### Question 3 (2 points)

If you run `df['GPA'] * 5`, the result is:

A. A Python list  
B. A NumPy array  
C. A pandas Series  
D. A new DataFrame  

**Explain your choice.**

### Question 4 (4 points)

Does multiplying the GPA column by 5 change:
- The number of rows?
- The number of columns?

**Explain why or why not for each.**

---
# PART 1 - Identifying the Stopping Potential in Photoelectric Spectra
`20 points`

An applied voltage can be used to oppose the motion of photoemitted electrons. As this **stopping potential** $V_s$ is increased, the measured photocurrent decreases; when the current is reduced to approximately zero, no electrons reach the anode. The stopping potential is therefore related to the maximum kinetic energy of the emitted electrons by
$$
K_{\max} = eV_s,
$$
where $e$ is the elementary charge.

In this section, you will visualize photoelectric spectra (photocurrent vs. applied voltage) and identify the experimental signature of the stopping potential.

### CODE: Load and Plot a Spectrum

For **one** spectrum file:

1. Load the CSV into a DataFrame named `data` using `pd.read_csv(...)`
2. Verify the DataFrame contains expected columns (show `data.head()`)
3. Produce a scatter plot of **current vs. voltage**
4. Label axes with units, add a title, and include a grid

> **Tip:** Reference columns as `data['column_name']` rather than hard-coding arrays.

**Available data files:**
- `data/Ca_200nm.csv`
- `data/Ca_250nm.csv`
- `data/Ca_300nm.csv`
- `data/Ca_350nm.csv`

In [ ]:
# ============================================
# SUBGOAL: Load and inspect data
# ============================================

# Load one of the spectrum files
data = pd.read_csv('data/Ca_200nm.csv')

# Display the first few rows to see column names and data structure
data.head()

In [ ]:
# ============================================
# SUBGOAL: Visualize and interpret
# ============================================

# Set the column names based on what you see in data.head()
# The actual column names are 'Voltage (V)' and 'Current (pA)'
xcol = 'Voltage (V)'
ycol = 'Current (pA)'

plt.figure()
plt.scatter(data[xcol], data[ycol])
plt.xlabel(xcol)
plt.ylabel(ycol)
plt.title('Photoelectric spectrum: current vs voltage')
plt.grid(True)
plt.show()

### Question 1 (3 points)

List the column names in `data` and briefly state the physical quantity and units represented by each.

### Question 2 (3 points)

What does a single row in the dataset correspond to experimentally? Why is it reasonable to treat each row as an independent measurement?

### Question 3 (5 points)

Describe the behavior of the photocurrent as the applied voltage increases. Identify the voltage range over which the current rapidly decreases and explain its physical significance.

### Question 4 (5 points)

From your plot, estimate the stopping potential $V_s$. Using
$$
K_{\max} = eV_s,
$$
report the corresponding maximum kinetic energy in electronvolts.

### Question 5 (4 points)

If the light intensity increases but the frequency is unchanged (and still above threshold), the stopping potential $V_s$ is expected to:

A. Increase  
B. Decrease  
C. Remain approximately unchanged  
D. Become undefined  

**Explain briefly using the physical meaning of $V_s$.**

---
# PART 2 - Processing the Spectrum
`20 points`

In this section, you will extract the **stopping potential** $V_s$ from the spectrum using a reproducible numerical procedure.

The analysis involves three operations:
1. **Filtering** - isolating the portion of the spectrum relevant for analysis
2. **Fitting** - performing linear regression on the nonzero-current region
3. **Reduction** - extracting $V_s$ from the x-intercept of the fitted line

Since $K_{\max} = eV_s$, when we measure voltage in volts the stopping potential equals $K_{\max}$ in electronvolts.

### CODE: Filter, Fit, and Extract $V_s$

Starting from the DataFrame `data` created in Part 1:

1. **Filter** the data to isolate the nonzero-current region used for fitting
   - Due to fluctuations, find where current approaches zero and discard points beyond that
2. **Fit** current vs. voltage using `np.polyfit(..., deg=1)`
3. **Plot** the filtered data and regression line on the same axes
4. **Extract** the x-intercept using `np.roots()` to find $V_s$

In [ ]:
# ============================================
# SUBGOAL: Process/transform data (filtering)
# ============================================

# Filter data to keep only rows where current is above a threshold
# This removes the noisy region near zero current
current_threshold = 0.5  # pA - adjust if needed based on your data

# YOUR CODE: Create data_filtered by selecting rows where current > threshold
# Hint: data_filtered = data[data['Current (pA)'] > current_threshold]


In [ ]:
# ============================================
# SUBGOAL: Fit model to data
# ============================================

# Perform linear regression: current = slope * voltage + intercept
# np.polyfit returns [slope, intercept] for deg=1

# YOUR CODE: Use np.polyfit to get slope and intercept
# coeffs = np.polyfit(data_filtered['Voltage (V)'], data_filtered['Current (pA)'], deg=1)
# slope = coeffs[0]
# intercept = coeffs[1]


In [ ]:
# ============================================
# SUBGOAL: Visualize and report
# ============================================

# Plot the filtered data and regression line
plt.figure(figsize=(8, 5))

# Plot the filtered data points
plt.scatter(data_filtered['Voltage (V)'], data_filtered['Current (pA)'], 
            label='Filtered data', alpha=0.7)

# Create the trendline
x_trendline = np.linspace(0, max(data_filtered['Voltage (V)']), 200)
y_trendline = slope * x_trendline + intercept
plt.plot(x_trendline, y_trendline, 'r-', label='Linear fit')

plt.xlabel('Voltage (V)')
plt.ylabel('Current (pA)')
plt.title('Linear fit to photoelectric spectrum')
plt.legend()
plt.grid(True)
plt.show()

# Compute the x-intercept (stopping potential)
# np.roots gives the roots of a polynomial (where y == 0)
# For a linear equation, there is only one root - access it with [0]
roots = np.roots([slope, intercept])
stopping_potential = roots[0]

print(f'Stopping Potential V_s = {stopping_potential:.3f} V')
print(f'Maximum Kinetic Energy K_max = {stopping_potential:.3f} eV')

### Question 1 (5 points)

Explain why data points near zero current should be excluded when performing a linear regression to determine $V_s$.

### Question 2 (5 points)

Report the stopping potential obtained from your fit. How does this value compare to your visual estimate from Part 1?

### Question 3 (10 points)

If this analysis were repeated using light of a **longer wavelength**, how would you expect $V_s$ to change? Justify your answer physically.

---
# PART 3 - Automating the Analysis
`20 points`

To determine how electron kinetic energy depends on the frequency of incident light, we must apply the analysis from Part 2 consistently across multiple spectra. Rather than repeating the same operations manually, you will encapsulate the procedure in a reusable function.

## 3a - Processing One Spectrum
`10 points`

### CODE: Write a Reusable Function

Write a function `process_spectrum(filename, plot=True)` that:

1. Loads the spectrum from `filename`
2. Extracts the wavelength and frequency from the data
3. Determines $V_s$ using the regression procedure from Part 2
4. If `plot=True`, produces a plot of the spectrum with the fitted line

The function must return (in this order):
```python
wavelength_nm, frequency_Hz, V_s
```

**Test your function** on several spectra corresponding to different wavelengths.

In [ ]:
def process_spectrum(filename, plot=True):
    """
    Process a photoelectric spectrum file and extract the stopping potential.
    
    Parameters
    ----------
    filename : str
        Path to the CSV file containing the spectrum data.
    plot : bool, optional
        If True, display a plot of the data with the fitted line. Default is True.
    
    Returns
    -------
    wavelength_nm : float
        Wavelength of the incident light in nanometers.
    frequency_Hz : float
        Frequency of the incident light in Hz.
    V_s : float
        Stopping potential in volts (equals K_max in eV).
    """
    # ============================================
    # SUBGOAL: Load and inspect data
    # ============================================
    # YOUR CODE: Load the CSV file
    
    # YOUR CODE: Extract wavelength and frequency from the data
    # (Hint: these are constant for each file, so you can take the first value)
    
    # ============================================
    # SUBGOAL: Process/transform data
    # ============================================
    # YOUR CODE: Filter the data to keep only nonzero current region
    
    # ============================================
    # SUBGOAL: Fit model to data
    # ============================================
    # YOUR CODE: Perform linear regression
    
    # YOUR CODE: Calculate the x-intercept (stopping potential)
    
    # ============================================
    # SUBGOAL: Visualize and report
    # ============================================
    if plot:
        # YOUR CODE: Create the plot
        pass
    
    return wavelength_nm, frequency_Hz, V_s

In [ ]:
# Test your function on multiple spectrum files

# Example:
# wavelength, frequency, Vs = process_spectrum('data/Ca_200nm.csv')
# print(f'Wavelength: {wavelength} nm, Frequency: {frequency:.2e} Hz, V_s: {Vs:.3f} V')

### Question (10 points)

Briefly explain why returning numerical values from `process_spectrum` is preferable to computing them only inside a plotting routine.

## 3b - Scaling Up the Analysis
`10 points`

In this section, you will apply your analysis to the entire collection of spectra. A helper function has been provided to iterate over files and aggregate results into a single table.

### CODE: Process All Spectra

In [ ]:
from helper import process_all_data

Run `process_all_data` on your spectra folder (ensure the folder contains only CSV files for this assignment). Store the result in a variable and display the resulting DataFrame.

In [ ]:
# Process all spectra and store results
# results = process_all_data('data', process_spectrum)
# results

### Question 1 (3 points)

Why is it advantageous to automate the analysis across all spectra rather than processing each spectrum individually?

### Question 2 (3 points)

Describe the trend you observe in the stopping potential as the wavelength of the incident light changes.

### Question 3 (4 points)

Explain how this trend supports or contradicts the classical wave description of light.

---
# PART 4 - Light Energy Versus Frequency
`20 points`

If light delivers energy to electrons in discrete packets, the energy carried by a single photon is
$$
E_{\text{photon}} = h\nu,
$$
where $h$ is Planck's constant and $\nu$ is the frequency of the incident light.

When a photon ejects an electron from the metal, this energy is partitioned between:
1. The energy required to remove the electron (the **work function** $\phi$)
2. The kinetic energy carried away by the emitted electron

Energy conservation gives:
$$
h\nu = K_{\max} + \phi
$$

Using $K_{\max} = eV_s$ and rearranging:
$$
V_s = \frac{h}{e}\nu - \frac{\phi}{e}
$$

This predicts that a plot of $V_s$ vs. $\nu$ should be **linear** with:
- **Slope** = $h/e$ (Planck's constant divided by elementary charge)
- **y-intercept** = $-\phi/e$ (negative work function in eV)

Since our stopping potential is already measured in volts (equivalent to eV per elementary charge), we can extract $h$ in eV$\cdot$s directly from the slope.

### CODE: Determine Planck's Constant and Work Function

Using the DataFrame from Part 3:
1. Perform a linear regression of $V_s$ vs. frequency (in Hz)
2. Plot the data with the fitted regression line
3. Extract the slope and intercept
4. Convert the slope from eV$\cdot$s to J$\cdot$s
5. Determine the work function from the intercept

In [ ]:
# ============================================
# SUBGOAL: Fit model to data
# ============================================

# YOUR CODE: Extract frequency and V_s columns from results DataFrame
# YOUR CODE: Perform linear regression using np.polyfit


In [ ]:
# ============================================
# SUBGOAL: Visualize and report
# ============================================

# YOUR CODE: Plot V_s vs frequency with fitted line
# Label axes appropriately!


In [ ]:
# ============================================
# SUBGOAL: Extract physical constants
# ============================================

# Convert slope from eV*s to J*s
# 1 eV = 1.602e-19 J
# h_eVs = slope  # slope is h/e in units of V*s = eV*s / e = eV*s
# h_Js = h_eVs * 1.602e-19

# The work function (in eV) is the negative of the y-intercept
# phi_eV = -intercept

# Accepted value: h = 6.626e-34 J*s = 4.136e-15 eV*s

# YOUR CODE: Calculate and print your values


### Question 1 (5 points)

The proportionality constant relating energy and frequency obtained from your fit is a well-known physical constant. Identify this constant and describe its physical significance.

### Question 2 (5 points)

Compare your calculated value of Planck's constant to the accepted value ($h = 6.626 \times 10^{-34}$ J$\cdot$s). What is the percent error?

### Question 3 (5 points)

What value do you obtain for the work function of calcium? The accepted value is approximately 2.9 eV. How does your value compare?

### Question 4 (5 points)

Discuss at least two sources of systematic or experimental error that could affect the slope or intercept of your fit.

---
# REFLECTION
`10 points`

### Question 1 (5 points)

How did the use of `pandas` and user-defined functions enable reproducible and scalable data analysis in this experiment? Describe how similar computational workflows could support data analysis or modeling tasks you might encounter as a practicing chemist.

### Question 2 (5 points)

How do your experimental results from the photoelectric effect contradict predictions of the classical wave model of light? Cite specific features of your data (e.g., threshold behavior, linear trends) that support a particle-based description.

---
# References

1. Einstein, A. (1905). "On a Heuristic Viewpoint Concerning the Production and Transformation of Light." *Annalen der Physik*, 17(6), 132-148.

2. Millikan, R.A. (1916). "A Direct Photoelectric Determination of Planck's 'h'." *Physical Review*, 7(3), 355-388.

3. Atkins, P. & de Paula, J. (2014). *Physical Chemistry*, 10th ed. Oxford University Press. Chapter 7.

4. pandas documentation: https://pandas.pydata.org/docs/

5. NumPy documentation: https://numpy.org/doc/